In [ ]:
!pip install diffusers transformers accelerate peft torchmetrics fairlearn albumentations huggingface_hub -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 19.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd

# Load filtered metadata
dark_clean = pd.read_csv('/content/drive/MyDrive/dermatology-research/data/filtered/fst_v_vi_only/dark_skin_metadata.csv')

# Caption generation function
def create_caption(row):
    caption = f"clinical dermatology photograph of {row['Disease_label']}, "\
              f"a {row['Main_class']} condition, "\
              f"on {row['Fitzpatrick']} Indian dark skin, "\
              f"located on {row['Body_part']}"
    return caption

# Apply to all rows
dark_clean['caption'] = dark_clean.apply(create_caption, axis=1)

# Check first 3
for idx, row in dark_clean.head(3).iterrows():
    print(f"Image: {row['Image_name']}")
    print(f"Caption: {row['caption']}")
    print()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/dermatology-research/data/filtered/fst_v_vi_only/dark_skin_metadata.csv'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

# Load filtered metadata
dark_clean = pd.read_csv('/content/drive/MyDrive/dermatology-research/data/filtered/fst_v_vi_only/dark_skin_metadata.csv')

# Caption generation function
def create_caption(row):
    caption = f"clinical dermatology photograph of {row['Disease_label']}, "\
              f"a {row['Main_class']} condition, "\
              f"on {row['Fitzpatrick']} Indian dark skin, "\
              f"located on {row['Body_part']}"
    return caption

# Apply to all rows
dark_clean['caption'] = dark_clean.apply(create_caption, axis=1)

# Check first 3
for idx, row in dark_clean.head(3).iterrows():
    print(f"Image: {row['Image_name']}")
    print(f"Caption: {row['caption']}")
    print()

Image: IMG_0015.jpg
Caption: clinical dermatology photograph of Vitiligo, a Pigmentary Disorders condition, on FST 5 Indian dark skin, located on Genital Area (Pubic Region)

Image: IMG_0036.jpg
Caption: clinical dermatology photograph of Eczema, a Inflammatory Disorders condition, on FST 5 Indian dark skin, located on Palms

Image: IMG_0040.jpg
Caption: clinical dermatology photograph of Tinea Cruris, a Infectious Disorders condition, on FST 5 Indian dark skin, located on Trunk Buttocks (Gluteal Region)



In [ ]:
# Save updated metadata with captions
output_path = '/content/drive/MyDrive/dermatology-research/data/filtered/fst_v_vi_only/dark_skin_metadata_with_captions.csv'
dark_clean.to_csv(output_path, index=False)

print(f"Saved! Total images with captions: {len(dark_clean)}")
print(f"\nSample caption count check:")
print(f"Captions generated: {dark_clean['caption'].notna().sum()}")

Saved! Total images with captions: 1310

Sample caption count check:
Captions generated: 1310


In [ ]:
from PIL import Image
import os

src = '/content/drive/MyDrive/dermatology-research/data/filtered/fst_v_vi_only/images'
dst = '/content/drive/MyDrive/dermatology-research/data/filtered/fst_v_vi_only/images_512'

os.makedirs(dst, exist_ok=True)

resized = 0
failed = 0

for img_name in dark_clean['Image_name']:
    src_path = os.path.join(src, img_name)
    dst_path = os.path.join(dst, img_name)

    try:
        img = Image.open(src_path).convert('RGB')
        img = img.resize((512, 512), Image.LANCZOS)
        img.save(dst_path, quality=95)
        resized += 1
    except Exception as e:
        print(f"Failed: {img_name} → {e}")
        failed += 1

print(f"Resized: {resized}")
print(f"Failed: {failed}")

Resized: 1310
Failed: 0


In [ ]:
import json
import os

# Output path
jsonl_path = '/content/drive/MyDrive/dermatology-research/data/filtered/fst_v_vi_only/metadata.jsonl'

with open(jsonl_path, 'w') as f:
    for idx, row in dark_clean.iterrows():
        entry = {
            "file_name": row['Image_name'],
            "text": row['caption']
        }
        f.write(json.dumps(entry) + '\n')

print(f"Created metadata.jsonl with {len(dark_clean)} entries")

# Verify first 2 lines
with open(jsonl_path, 'r') as f:
    for i, line in enumerate(f):
        if i < 2:
            print(json.loads(line))

Created metadata.jsonl with 1310 entries
{'file_name': 'IMG_0015.jpg', 'text': 'clinical dermatology photograph of Vitiligo, a Pigmentary Disorders condition, on FST 5 Indian dark skin, located on Genital Area (Pubic Region)'}
{'file_name': 'IMG_0036.jpg', 'text': 'clinical dermatology photograph of Eczema, a Inflammatory Disorders condition, on FST 5 Indian dark skin, located on Palms'}


In [ ]:
from google.colab import userdata
from huggingface_hub import login

# Login to HuggingFace
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print("HuggingFace login successful!")

HuggingFace login successful!


In [ ]:
from diffusers import StableDiffusionPipeline
import torch

print("Loading SD 2.0...")
pipeline = StableDiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-2-1-base",
    torch_dtype=torch.float16,
    safety_checker=None
)
pipeline.to("cuda")
print("SD 2.0 loaded successfully!")
print(f"Device: {next(pipeline.unet.parameters()).device}")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Loading SD 2.0...


Couldn't connect to the Hub: 404 Client Error. (Request ID: Root=1-69b8487f-07d119184a3fcb4f284483e5;fc7217b5-37bc-40e8-9863-d2f761bbbc32)

Repository Not Found for url: https://huggingface.co/api/models/stabilityai/stable-diffusion-2-1-base.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication.
Will try to load from local cache.


OSError: Cannot load model stabilityai/stable-diffusion-2-1-base: model is not cached locally and an error occurred while trying to fetch metadata from the Hub. Please check out the root cause in the stacktrace above.

In [ ]:
import os

# Configure git
!git config --global user.email "srushtikulkarni528@gmail.com"
!git config --global user.name "Srushti Kulkarni"

# Go to your cloned repo
os.chdir('/content/drive/MyDrive/dermatology-research/code')

# Copy notebook to repo
!cp '/content/drive/MyDrive/dermatology-research/notebooks/03_lora_finetuning.ipynb' .

# Push to GitHub
!git add .
!git commit -m "Add LoRA fine-tuning notebook with caption generation and dataset preparation"
!git push

cp: cannot stat '/content/drive/MyDrive/dermatology-research/notebooks/03_lora_finetuning.ipynb': No such file or directory
On branch main

Initial commit

nothing to commit (create/copy files and use "git add" to track)
error: src refspec refs/heads/main does not match any
error: failed to push some refs to 'https://github.com/kulkarni997/dermatology-skin-bias-research.git'


In [ ]:
!git checkout -b main